In [28]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [29]:
df = pd.read_csv("/content/construction_site_risk_dataset.csv")
df.head()
df.shape

(1000, 15)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   site_id                      1000 non-null   object 
 1   helmet_compliance            1000 non-null   float64
 2   ppe_compliance               1000 non-null   float64
 3   equipment_condition          1000 non-null   float64
 4   equipment_age_years          1000 non-null   float64
 5   safety_delay                 1000 non-null   object 
 6   worker_count                 1000 non-null   int64  
 7   site_type                    1000 non-null   object 
 8   weather_condition            1000 non-null   object 
 9   supervisor_experience_years  1000 non-null   float64
 10  incident_history_count       1000 non-null   int64  
 11  inspection_frequency_days    1000 non-null   float64
 12  scaffolding_used             1000 non-null   object 
 13  night_shift        

In [31]:
df.describe()

,helmet_compliance,ppe_compliance,equipment_condition,equipment_age_years,worker_count,supervisor_experience_years,incident_history_count,inspection_frequency_days,risk_score
count,1000.000000,1000.000000,1000.00000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,63.937700,62.781300,65.27680,5.57140,100.617000,8.685300,1.185000,20.476100,49.804000
std,20.728205,22.130634,19.49703,5.35465,56.333949,4.308154,1.124295,19.674836,19.100788
min,5.000000,5.000000,5.00000,0.00000,5.000000,1.200000,0.000000,1.000000,0.000000
25%,49.700000,48.600000,51.80000,1.70000,54.000000,5.500000,0.000000,5.700000,35.700000
50%,65.100000,64.250000,65.70000,4.05000,98.000000,8.000000,1.000000,14.200000,49.900000
75%,78.000000,77.800000,79.50000,7.40000,149.000000,11.100000,2.000000,29.025000,63.400000
max,100.000000,100.000000,100.00000,25.00000,200.000000,30.000000,6.000000,90.000000,100.000000


In [32]:
df.isnull().sum()

,0
site_id,0
helmet_compliance,0
ppe_compliance,0
equipment_condition,0
equipment_age_years,0
safety_delay,0
worker_count,0
site_type,0
weather_condition,0
supervisor_experience_years,0


In [34]:
df.columns

Index(['site_id', 'helmet_compliance', 'ppe_compliance', 'equipment_condition',
       'equipment_age_years', 'safety_delay', 'worker_count', 'site_type',
       'weather_condition', 'supervisor_experience_years',
       'incident_history_count', 'inspection_frequency_days',
       'scaffolding_used', 'night_shift', 'risk_score'],
      dtype='object')

In [35]:
X = df.drop(columns=["site_id", "risk_score"])
y = df["risk_score"]

In [36]:
categorical_features = [
    "safety_delay",
    "site_type",
    "weather_condition",
    "scaffolding_used",
    "night_shift"
]

numerical_features = [
    "helmet_compliance",
    "ppe_compliance",
    "equipment_condition",
    "equipment_age_years",
    "worker_count",
    "supervisor_experience_years",
    "incident_history_count",
    "inspection_frequency_days"
]

In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

In [38]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [40]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['safety_delay', 'site_type',
                                                   'weather_condition',
                                                   'scaffolding_used',
                                                   'night_shift']),
                                                 ('num', 'passthrough',
                                                  ['helmet_compliance',
                                                   'ppe_compliance',
                                                   'equipment_condition',
                                                   'equipment_age_years',
                                                   'worker_count',
                                                   'supervisor_experience_years',
                                                   'incident_history_count',
                                                   'inspection_frequency_days'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [41]:
y_pred = model.predict(X_test)

In [42]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae,2))
print("RMSE:", round(rmse,2))
print("R² :", round(r2,4))

MAE : 5.2
RMSE: 6.45
R² : 0.8988


In [43]:
joblib.dump(model, "risk_model.pkl")

print("✅ Agent 2 Model Saved Successfully!")

✅ Agent 2 Model Saved Successfully!


In [44]:
new_site = pd.DataFrame([{
    "helmet_compliance": 80,
    "ppe_compliance": 75,
    "equipment_condition": 60,
    "equipment_age_years": 5,
    "safety_delay": "Medium",
    "worker_count": 120,
    "site_type": "Bridge",
    "weather_condition": "Rain",
    "supervisor_experience_years": 8,
    "incident_history_count": 2,
    "inspection_frequency_days": 15,
    "scaffolding_used": "Yes",
    "night_shift": "No"
}])

predicted_score = model.predict(new_site)[0]

if predicted_score < 40:
    level = "Low"
elif predicted_score < 70:
    level = "Medium"
else:
    level = "High"

print("Predicted Risk Score:", round(predicted_score,2))
print("Risk Level:", level)

Predicted Risk Score: 50.93
Risk Level: Medium
